# 04 · RAG with privacy
**The limitation this fixes:** the agent can't answer from our leaflets or check-in notes, and it would happily echo identifiers.

| Feature | Framework piece |
|---|---|
| Load, chunk, embed, index | `Document`, `RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`, `InMemoryVectorStore` |
| Retrieval | just another `@tool` |
| Show sources | `response_format="content_and_artifact"` |
| Patient A can't see patient B's notes | filter built from `runtime.context` |
| Hide MRNs and phone numbers, even inside retrieved notes | `PIIMiddleware` (from 02) |
| Audit log of every document search | `audit_log` (from 02) |
| Everything at once | the middleware stack from 02 + 03 |

In [ ]:
# Setup: the same cell at the top of every notebook
from dotenv import load_dotenv
load_dotenv("../.env", override=True)   # OPENAI_API_KEY + LANGSMITH_* (set LANGSMITH_TRACING=true to see traces)

MODEL = "openai:gpt-4o-mini"           # swap the provider here, e.g. "anthropic:claude-sonnet-4-6"

## What we built so far

In [ ]:
from common import (CareContext, SYSTEM_PROMPT, CARE_PROMPT, DATA, FALLBACK_MODEL,
                    get_care_plan, log_reading, log_symptom, get_recent_readings, escalate_to_nurse,   # 01, 03
                    verify_identity, list_free_slots, book_appointment, patients_needing_attention,   # 02
                    pii_middleware, audit_log, AUDIT_LOG, pick_tools_and_model, make_care_prompt,      # 02
                    dose_guard,                                                                        # 03
                    ask, resume, show_state, talk)
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

## 🔓 Warm-up: catch it making things up
Our clinic's leaflets are in `data/leaflets/`. The bot below **can't read them**.
**In pairs:** find a question it answers *confidently* and *differently from the leaflet*. Fastest pair wins.
Ask it anything about measuring blood pressure, glucose or medicines. Peek at the leaflets in the next cell to check its answers.

In [ ]:
no_docs = create_agent(MODEL, tools=[get_care_plan], system_prompt=SYSTEM_PROMPT, context_schema=CareContext)

talk(no_docs, thread="hunt")   # 🛠 type your questions here

In [ ]:
# The truth, according to OUR clinic. Compare with the answer above.
for leaflet in sorted((DATA / "leaflets").glob("*.md")):
    print(leaflet.read_text())

In [ ]:
MY_QUESTION = "At what blood pressure reading should I contact my care team the same day?"   # 🛠 paste your best catch
ask(no_docs, MY_QUESTION)

**Why this matters:** the answer sounds right and is often *generally* true, but it isn't **our clinic's** rule.
A patient can't tell the difference. That's the gap RAG closes.

## 1. Load the documents
Leaflets are **public**. Check-in notes are **private** and tagged with the patient they belong to.

In [ ]:
from langchain_core.documents import Document

docs = [Document(p.read_text(), metadata={"source": p.name, "scope": "public"})
        for p in sorted((DATA / "leaflets").glob("*.md"))]
docs += [Document(p.read_text(), metadata={"source": p.name, "scope": "private", "patient_token": p.stem})
         for p in sorted((DATA / "notes").glob("*.md"))]

assert docs, "data/ is empty"
[d.metadata for d in docs]

## 2. Chunk, embed, index

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
len(chunks)

## 3. Retrieval is just a tool
The **filter** is built from `runtime.context`, inside the tool. The model only gets to choose the `query`.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool(response_format="content_and_artifact")
def search_documents(query: str, runtime: ToolRuntime[CareContext]):
    """Search approved patient leaflets and this patient's own check-in notes."""
    me = runtime.context.patient_token

    def allowed(doc):   # public leaflets + only MY notes
        return doc.metadata["scope"] == "public" or doc.metadata.get("patient_token") == me

    found = vector_store.similarity_search(query, k=3, filter=allowed)
    text = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in found)
    return text or "Nothing found in the documents.", [d.metadata for d in found]   # (for the model, for us)

## 4. Plug it into the full stack
Every hook below came from **02** or **03**. The only new things are one tool and two prompt rules.
`PIIMiddleware` now matters even more: the check-in notes we retrieve contain MRNs and phone numbers.

In [ ]:
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ModelFallbackMiddleware,
    SummarizationMiddleware, ToolCallLimitMiddleware, ToolRetryMiddleware)

RAG_PROMPT = CARE_PROMPT + """
- For how-to questions, or what the doctor noted or planned at past check-ins,
  use search_documents and cite the [source].
- If the documents don't contain the answer, say "That's not in my documents." Don't guess."""

middleware = [
    dose_guard,                                                   # 03: refuse dose questions
    *pii_middleware(),                                            # 02: redact MRNs and phones
    SummarizationMiddleware(MODEL, trigger=("messages", 40), keep=("messages", 20)),
    make_care_prompt(RAG_PROMPT),                                 # 02: prompt by role and language
    pick_tools_and_model,                                         # 02: tools by role and identity
    ModelFallbackMiddleware(FALLBACK_MODEL),
    ModelCallLimitMiddleware(run_limit=10),
    ToolCallLimitMiddleware(run_limit=10),
    ToolRetryMiddleware(max_retries=2),
    audit_log,                                                    # 02: who touched what
    HumanInTheLoopMiddleware(interrupt_on={"escalate_to_nurse": True}),   # 03: nurse approves
]

tools = [get_care_plan, log_reading, log_symptom, get_recent_readings, escalate_to_nurse,
         search_documents,                                        # 👈 the only new tool
         verify_identity, list_free_slots, book_appointment, patients_needing_attention]
agent = create_agent(MODEL, tools=tools, context_schema=CareContext,
                     checkpointer=InMemorySaver(), middleware=middleware)

ask(agent, "How should I measure my blood pressure at home?", thread="rag")

🧾 **Receipt:** the artifact holds the sources of the chunks the answer came from (the model never sees it).

In [ ]:
last_tool_message = [m for m in agent.get_state({"configurable": {"thread_id": "rag"}}).values["messages"]
                     if m.type == "tool"][-1]
last_tool_message.artifact

## 🛠 Back to your warm-up question
Same question, now with the leaflets. Does it match the leaflet, and does it cite the `[source]`?

In [ ]:
ask(agent, MY_QUESTION, thread="hunt")

## ❌ Failure case: Sara asks for Omar's notes
The retriever literally cannot return them: the filter came from Sara's context.

In [ ]:
ask(agent, "What do Omar Demo's check-in notes say about his knee?", thread="snoop")
show_state(agent, thread="snoop")

## PII: the MRN never reaches the model
`PIIMiddleware` from 02 at work. (The tool result also contained Sara's MRN and phone from her notes. Look for `[REDACTED_...]`.)

In [ ]:
ask(agent, "My MRN is MRN-100001. What did my doctor plan at my last check-in?", thread="pii")
show_state(agent, thread="pii")

🧾 **Receipt:** the audit log.

In [ ]:
AUDIT_LOG

## Where this lives in the backend
- `app/rag.py: loading, chunking, the vector store and `search_documents``
- `app/middleware.py`: `build_middleware()`, the same stack as above

Same primitives, just split into files. The notebook never imports `app/`.